# Phase 3: Spatial matching of neighbourhoods and stations

Each neighbourhood is linked to its nearest railway station, so that station characteristics (code, name, type) can be added. The station code is later used to add disruption data (Phase 4–5).


In [15]:
from pathlib import Path
import pandas as pd
import geopandas as gpd

PROCESSED = Path('data/cleaned_data')

buurten = pd.read_pickle(PROCESSED / 'buurten_schoon.pkl')
stations = pd.read_pickle(PROCESSED / 'stations.pkl')
buurten_kaart = gpd.read_file(PROCESSED / 'buurten_kaart.gpkg')

In [16]:
# Create a GeoDataFrame with the centroids of the neighborhoods
buurt_punten = buurten_kaart[buurten_kaart['buurtcode'].isin(buurten['buurtcode'])].copy()
buurt_punten['geometry'] = buurt_punten.centroid

print(len(buurt_punten))
buurt_punten.head()



11221


,buurtcode,buurtnaam,gemeentenaam,geometry
94,BU00140000,Binnenstad-Noord,Groningen,POINT (233655.736 582070.787)
95,BU00140001,Binnenstad-Zuid,Groningen,POINT (233834.594 581588.756)
96,BU00140002,Binnenstad-Oost,Groningen,POINT (234292.167 582097.291)
97,BU00140003,Binnenstad-West,Groningen,POINT (233195.739 581682.521)
99,BU00140005,Hortusbuurt-Ebbingekwartier,Groningen,POINT (233458.532 582523.152)


In [17]:
# Create a GeoDataFrame with the points of the stations
# Stap 1: kolommen een duidelijke naam geven
stations = stations.rename(columns={'code': 'station_code', 'name_long': 'station_naam', 'type': 'station_type'})

# Stap 2: van de coördinaten puntjes maken (eerst lengtegraad, dan breedtegraad)
punten = gpd.points_from_xy(stations['geo_lng'], stations['geo_lat'])

# Stap 3: een kaart-tabel maken met die puntjes; de getallen zijn in graden (EPSG:4326)
station_punten = gpd.GeoDataFrame(stations, geometry=punten, crs='EPSG:4326')

# Stap 4: omzetten van graden naar meters, zodat het past bij de buurtkaart
station_punten = station_punten.to_crs('EPSG:28992')

# Stap 5: alleen de kolommen houden die we nodig hebben
station_punten = station_punten[['station_code', 'station_naam', 'station_type', 'geometry']]

station_punten.head()

,station_code,station_naam,station_type,geometry
0,HT,'s-Hertogenbosch,knooppuntIntercitystation,POINT (148529.229 411304.081)
1,HTO,'s-Hertogenbosch Oost,stoptreinstation,POINT (150239.092 412422.932)
2,HDE,'t Harde,stoptreinstation,POINT (189457.906 491380.869)
3,ATN,Aalten,stoptreinstation,POINT (236958.504 437654.779)
4,AC,Abcoude,stoptreinstation,POINT (127005.741 476800.771)


In [18]:
#koppel de buurten aan de dichtstbijzijnde stations

# Zoek voor elke buurt het dichtstbijzijnde station en plak de stationsgegevens erachter
gekoppeld = gpd.sjoin_nearest(buurt_punten, station_punten, how='left')

# Controle: er moeten nog steeds 11221 buurten zijn
print(len(gekoppeld))
gekoppeld.head()





11221


,buurtcode,buurtnaam,gemeentenaam,geometry,index_right,station_code,station_naam,station_type
94,BU00140000,Binnenstad-Noord,Groningen,POINT (233655.736 582070.787),154,GN,Groningen,knooppuntIntercitystation
95,BU00140001,Binnenstad-Zuid,Groningen,POINT (233834.594 581588.756),154,GN,Groningen,knooppuntIntercitystation
96,BU00140002,Binnenstad-Oost,Groningen,POINT (234292.167 582097.291),154,GN,Groningen,knooppuntIntercitystation
97,BU00140003,Binnenstad-West,Groningen,POINT (233195.739 581682.521),154,GN,Groningen,knooppuntIntercitystation
99,BU00140005,Hortusbuurt-Ebbingekwartier,Groningen,POINT (233458.532 582523.152),156,GNN,Groningen Noord,stoptreinstation


In [ ]:
#wijzig de station_type kolom zodat we kunnen filteren op grote stations
grote_types = ['megastation', 'knooppuntIntercitystation', 'intercitystation']

gekoppeld['station_groot'] = gekoppeld['station_type'].isin(grote_types)

gekoppeld['station_groot'].value_counts()



station_groot
False    8677
True     2544
Name: count, dtype: int64

In [20]:
# Stap 1: uit de koppeling alleen de kolommen pakken die we nodig hebben
station_info = gekoppeld[['buurtcode', 'station_code', 'station_naam', 'station_type', 'station_groot']]

# Stap 2: de stationsgegevens aan de schone buurtentabel plakken, via de buurtcode
buurten_met_station = buurten.merge(station_info, on='buurtcode', how='left')

# Stap 3: controleren of het goed ging
print(buurten_met_station.shape)                                   # moet (11221, 17) zijn
print('Buurten zonder station:', buurten_met_station['station_code'].isna().sum())   # moet 0 zijn

# Stap 4: opslaan, zodat we het in de volgende fase kunnen gebruiken
buurten_met_station.to_pickle(PROCESSED / 'buurten_met_station.pkl')

(11221, 17)
Buurten zonder station: 0
